# Survival

> SURVIVE!

In [ ]:
#| default_exp survival

In [ ]:
#| export
import torch, torch.nn as nn, lightning.pytorch as pl, warnings, math, numpy as np

from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts
from sleepjepa.loss import nll_logistic_hazard

from sksurv.metrics import (
    concordance_index_ipcw,
    brier_score,
    cumulative_dynamic_auc,
    integrated_brier_score,
)
from sksurv.util import Surv

In [ ]:
#| export
def discretize_time(time_days, bin_years=0.25, min_year=0.0, max_year=10.0):
    """
    Discretize a tensor of observed times (in days) into bin indices for discrete-time survival.

    Args:
        time_days (Tensor): observed times in days (event or censoring).
        bin_years (float): width of bins in years (e.g., 0.25 = quarter-year).
        min_year (float): lower bound of the window in years.
        max_year (float): upper bound of the window in years.

    Returns:
        bin_idx (Tensor): indices in [0, n_bins-1] for each input time
    """
    days_per_bin = bin_years * 365.25
    min_days = min_year * 365.25
    max_days = max_year * 365.25

    time_days = torch.clamp(time_days, min=min_days, max=max_days)
    
    idx = torch.ceil(time_days / days_per_bin) - 1
    idx = idx.long()

    n_bins = int(np.ceil((max_days - min_days) / days_per_bin))
    idx = torch.clamp(idx, min=0, max=n_bins - 1)
    return idx

def make_cuts_days(min_year: float,
                   max_year: float,
                   bin_years: float):
    """
    Build right-edges of bins (in DAYS) for the discrete-time hazard model.

    Returns:
        cuts_days (np.ndarray): shape [J], right edges in days.
    """
    edges_years = np.arange(min_year + bin_years,
                            max_year + 1e-9,
                            bin_years)
    return edges_years * 365.25

def hazards_to_survival(preds):
    preds = np.exp(np.log(1 - preds + 1e-8).cumsum(axis=1))
    return preds

def hazards_to_survival_torch(preds):
    preds = torch.exp(torch.log(1 - preds + 1e-8).cumsum(dim=1))
    return preds

## Cox w Momentum

In [ ]:
#| export
class PatchTFTSurvivalDemo(pl.LightningModule):
    def __init__(self,
                learning_rate,
                train_size,
                batch_size,
                n_gpus,
                linear_probing_head,
                preloaded_model,
                evaluate_risk_years = [1,5,10],
                discretize_time=True,
                discrete_time_bins=0.25,
                time_range_years=(0,15),
                fine_tune=False,
                epochs=100,
                scheduler_type='OneCycle',
                optimizer_type='AdamW',
                weight_decay=0.001,
                final_weight_decay=1e-6,
                use_weight_decay_scheduler=False,
                demographic_embeddings={'age_bin_idx': 9,'bmi_bin_idx': 7,'gender': 2},
                age_mlp_hidden_size=32,
                demographics_only=False,
                scheduler_kwargs={},
                transforms=None,
                ):
        super().__init__()
        self.encoder = preloaded_model
        self.scheduler_type = scheduler_type
        self.weight_decay = weight_decay
        self.final_weight_decay = final_weight_decay
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.fine_tune = fine_tune
        self.optimizer_type = optimizer_type
        self.scheduler_kwargs = scheduler_kwargs
        self.discrete_time_bins = discrete_time_bins
        self.time_range_years = time_range_years
        self.discretize_time = discretize_time
        self.demographic_embeddings = demographic_embeddings
        self.demographics_only = demographics_only
        self.transforms = transforms
        if evaluate_risk_years is not None:
            warnings.warn("evaluate_risk_years is ignored when discretize_time=True. Risk will be evaluated at all bins.")
        self.new_time = torch.tensor(make_cuts_days(self.time_range_years[0], self.time_range_years[1], self.discrete_time_bins), dtype=torch.float32)
        self.evaluate_risk_years = self.new_time / 365.25
        
        if not self.fine_tune:
            self.encoder.freeze()
            if hasattr(self.encoder, 'pretrain'):
                setattr(self.encoder, 'pretrain', False)
                setattr(self.encoder.model, 'pretrain', False)
        if not self.demographics_only:
            self.feedforward = linear_probing_head
        else:
            self.feedforward = nn.Identity()

        self.time_bias = nn.Parameter(torch.zeros(len(self.new_time)))
        
        if "age_bin_idx" in demographic_embeddings:
            self.age_time_interaction = nn.Embedding(
                demographic_embeddings['age_bin_idx'],
                len(self.new_time)
            )
            nn.init.zeros_(self.age_time_interaction.weight)

        elif "age_s1" in demographic_embeddings:
            self.age_time_interaction = nn.Sequential(nn.Linear(1, age_mlp_hidden_size), nn.ReLU(), nn.Dropout(0.1), nn.Linear(age_mlp_hidden_size, len(self.new_time)))
        
        else:
            self.age_time_interaction = nn.Identity()

        self.demographic_effects = nn.ParameterDict()
        for k, v in demographic_embeddings.items():
            if k not in ['age_bin_idx', 'age_s1']:
                self.demographic_effects[k] = nn.Parameter(torch.zeros(v))


        self.train_ipcw_events = []
        self.train_ipcw_times = []
        self.events = []
        self.times = []
        self.preds = []
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x, y, time, demographics):
        x = self.encoder(x)
        if isinstance(x, tuple):
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        if self.time_range_years is not None:
            if y.ndim == 2:
                time = time.unsqueeze(1)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y)
            time = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
        time = discretize_time(time, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])

        if self.demographics_only:
            log_hz_k = self.time_bias
        else:
            log_hz_k = self.feedforward(x) + self.time_bias
        
        for i, k in enumerate(self.demographic_embeddings):
            if k == 'age_bin_idx':
                age_emb = self.age_time_interaction(demographics[:, i].to(int))
                log_hz_k = log_hz_k + age_emb
            elif k == 'age_s1':
                age_emb = self.age_time_interaction(demographics[:, i].unsqueeze(1))
                log_hz_k = log_hz_k + age_emb
            else:
                log_hz_k = log_hz_k + self.demographic_effects[k][demographics[:, i].to(int)].unsqueeze(1)
        return log_hz_k
          
    def on_train_batch_start(self, batch, batch_idx):
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd
    
    def predict_step(self, batch, batch_idx, dataloader_idx=0): 
        x, y, time, demographic_targets = batch
        log_hz_k = self(x, y, time, demographic_targets)
        return log_hz_k, y, time, demographic_targets

    def training_step(self, batch, batch_idx):
        x, y, time, demographic_targets = batch
        self.train_ipcw_events.append(y.bool().detach().cpu())
        self.train_ipcw_times.append(time.detach().cpu())

        if self.transforms is not None:
            batch = self.transforms(batch)
            x, y, time, demographic_targets = batch

        log_hz_k = self(x, y, time, demographic_targets)

        if self.time_range_years is not None:
            if y.ndim == 2:
                time = time.unsqueeze(1)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y)
            time_clamped = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
        else:
            time_clamped = time
        time_bins = discretize_time(time_clamped, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])
        
        if self.transforms is not None:
            time_pre_mix = self.train_ipcw_times[-1]
            time_pre_mix = torch.clamp(time_pre_mix, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
            time_bins_pre_mix = discretize_time(time_pre_mix, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])
            time_bins = torch.stack([time_bins.squeeze(-1), time_bins_pre_mix.squeeze(-1).to(time_bins.device)], dim=1)

        loss = nll_logistic_hazard(log_hz_k, y, time_bins)
        
        loss = loss.to(x.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y, time, demographic_targets = batch
        
        log_hz_k = self(x, y, time, demographic_targets)

        if self.time_range_years is not None:
            if y.ndim == 2:
                time = time.unsqueeze(1)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y)
            time_clamped = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
        else:
            time_clamped = time
        time_bins = discretize_time(time_clamped, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])
        loss = nll_logistic_hazard(log_hz_k, y, time_bins)

        loss = loss.to(x.device)
        log_hz_k = log_hz_k.to(x.device)
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        
        self.events.append(y.bool().detach().cpu())
        self.times.append(time.detach().cpu())
        self.preds.append(log_hz_k.detach())
        return loss

    def on_validation_epoch_end(self):

        preds = torch.cat(self.preds).squeeze()
        preds = torch.sigmoid(preds)
        
        og_device = preds.device.type

        preds = preds.cpu().to(torch.float32).numpy()
        events = torch.cat(self.events).squeeze().numpy()
        times = torch.cat(self.times).squeeze().numpy()
        new_time = self.new_time.cpu().numpy()
        evaluate_risk_years = self.evaluate_risk_years.cpu().numpy()
        
        preds = 1 - hazards_to_survival(preds)

        try:
            min_time, max_time = times.min(), times.max()
            if new_time.min() < min_time or new_time.max() > max_time:
                warnings.warn(f"Some evaluation times are outside the range of observed times. Clipping new_time from [{new_time.min()}, {new_time.max()}] to [{min_time}, {max_time}]")
                idx_ = np.where((new_time >= min_time) & (new_time <= max_time))[0]
                preds = preds[:, idx_]
                new_time = new_time[idx_]
                evaluate_risk_years = evaluate_risk_years[idx_]

            min_first_event_time = times[events == True].min()

            idx_ = np.where((new_time >= min_first_event_time))[0]
            preds = preds[:, idx_]
            new_time = new_time[idx_]
            evaluate_risk_years = evaluate_risk_years[idx_]
            
            tau = new_time[-1]
            print(f"Events shape: {events.shape}, sum: {events.sum()}")
            print(f"Times shape: {times.shape}, range: [{times.min()}, {times.max()}]")
            print(f"Predictions shape: {preds.shape}, range: [{preds.min()}, {preds.max()}]")
        except Exception as e:
            print(f"IPCW or time clipping error: {str(e)}")


        try:
            train_events = torch.cat(self.train_ipcw_events).squeeze().numpy()
            train_times = torch.cat(self.train_ipcw_times).squeeze().numpy()

            
            train_structured = Surv.from_arrays(event=train_events, time=train_times)
            val_structured = Surv.from_arrays(event=events, time=times)
        except Exception as e:
            print(f"Structured array creation error: {str(e)}")
        try:
            auc_all, mean_auc = cumulative_dynamic_auc(train_structured, val_structured, preds, new_time)
            auc_all = torch.tensor(auc_all, device=og_device)
            mean_auc = torch.tensor(mean_auc, device=og_device)
        except Exception as e:
            print(f"AUC calculation error: {str(e)}")
            auc_all = torch.tensor([0.0 for _ in evaluate_risk_years], device=og_device)
            mean_auc = torch.tensor(0.0, device=og_device)
        finally:
            self.log_dict({f"{y}year_auc_ipcw": score for y, score in zip(evaluate_risk_years, auc_all)}, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            self.log("mean_ipcw_auc", mean_auc, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        try:
            cindex = concordance_index_ipcw(train_structured, val_structured, preds[:,-1], tau=tau)[0]
            cindex = torch.tensor(cindex, device=og_device)
            self.log("cindex_ipcw", cindex, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"C-index calculation error: {str(e)}")
        try:
            preds = 1 - preds
            brier = integrated_brier_score(train_structured, val_structured, preds, new_time)
            brier = torch.tensor(brier, device=og_device)
            self.log("brier_score", brier, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"Brier calculation error: {str(e)}")
        try:
            all_score = mean_auc + (1 - brier) + cindex
            self.log("all_score", all_score, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"All score calculation error: {str(e)}")
        self.preds.clear()
        self.events.clear()
        self.times.clear()

    def on_validation_epoch_start(self):
        torch.cuda.empty_cache()

    def configure_optimizers(self):
        param_groups = [
            {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
             {
                'params': (self.time_bias)
            },
            {
                'params': (p for n, p in self.demographic_effects.named_parameters()) 
            },
            {
                'params': (p for n, p in self.age_time_interaction.named_parameters())
            }
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, total_steps=self.trainer.estimated_stepping_batches, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| export
class SurvivalDemo(pl.LightningModule):
    def __init__(self,
                learning_rate,
                train_size,
                batch_size,
                n_gpus,
                evaluate_risk_years = [1,5,10],
                discretize_time=True,
                discrete_time_bins=0.25,
                time_range_years=(0,15),
                fine_tune=False,
                epochs=100,
                scheduler_type='OneCycle',
                optimizer_type='AdamW',
                weight_decay=0.001,
                final_weight_decay=1e-6, 
                use_weight_decay_scheduler=False,
                demographic_embeddings={'age_bin_idx': 9,'bmi_bin_idx': 7,'gender': 2},
                age_mlp_hidden_size=32,
                scheduler_kwargs={},
                transforms=None,
                ):
        super().__init__()
        self.scheduler_type = scheduler_type
        self.weight_decay = weight_decay
        self.final_weight_decay = final_weight_decay
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.fine_tune = fine_tune
        self.optimizer_type = optimizer_type
        self.scheduler_kwargs = scheduler_kwargs
        self.discrete_time_bins = discrete_time_bins
        self.time_range_years = time_range_years
        self.discretize_time = discretize_time
        self.demographic_embeddings = demographic_embeddings
        self.transforms = transforms
        if evaluate_risk_years is not None:
            warnings.warn("evaluate_risk_years is ignored when discretize_time=True. Risk will be evaluated at all bins.")
        self.new_time = torch.tensor(make_cuts_days(self.time_range_years[0], self.time_range_years[1], self.discrete_time_bins), dtype=torch.float32)
        self.evaluate_risk_years = self.new_time / 365.25

        self.time_bias = nn.Parameter(torch.zeros(len(self.new_time)))
        
        if "age_bin_idx" in demographic_embeddings:
            self.age_time_interaction = nn.Embedding(
                demographic_embeddings['age_bin_idx'],
                len(self.new_time)
            )
            nn.init.zeros_(self.age_time_interaction.weight)

        elif "age_s1" in demographic_embeddings:
            self.age_time_interaction = nn.Sequential(nn.Linear(1, age_mlp_hidden_size), nn.ReLU(), nn.Dropout(0.1), nn.Linear(age_mlp_hidden_size, len(self.new_time)))
        
        else:
            self.age_time_interaction = nn.Identity()

        self.demographic_effects = nn.ParameterDict()
        for k, v in demographic_embeddings.items():
            if k not in ['age_bin_idx', 'age_s1']:
                self.demographic_effects[k] = nn.Parameter(torch.zeros(v))


        self.train_ipcw_events = []
        self.train_ipcw_times = []
        self.events = []
        self.times = []
        self.preds = []
        self.save_hyperparameters()

    def forward(self, x, y, time, demographics):
        if self.time_range_years is not None:
            if y.ndim == 2:
                # soft labels
                time = time.unsqueeze(1)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y) # censor if clipped
            time = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
        time = discretize_time(time, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])

        log_hz_k = self.time_bias
        
        for i, k in enumerate(self.demographic_embeddings):
            if k == 'age_bin_idx':
                age_emb = self.age_time_interaction(demographics[:, i].to(int))
                log_hz_k = log_hz_k + age_emb
            elif k == 'age_s1':
                age_emb = self.age_time_interaction(demographics[:, i].unsqueeze(1))
                log_hz_k = log_hz_k + age_emb
            else:
                log_hz_k = log_hz_k + self.demographic_effects[k][demographics[:, i].to(int)].unsqueeze(1)
        return log_hz_k
          
    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd
    
    def predict_step(self, batch, batch_idx, dataloader_idx=0): 
        x, y, time, demographic_targets = batch
        log_hz_k = self(x, y, time, demographic_targets)
        return log_hz_k, y, time, demographic_targets

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        x, y, time, demographic_targets = batch
        self.train_ipcw_events.append(y.bool().detach().cpu())
        self.train_ipcw_times.append(time.detach().cpu())
        
        if self.transforms is not None:
            batch = self.transforms(batch)
            x, y, time, demographic_targets = batch

        log_hz_k = self(x, y, time, demographic_targets)

        if self.time_range_years is not None:
            if y.ndim == 2:
                # soft labels
                time = time.unsqueeze(1)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y) # censor if clipped
            time_clamped = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
        else:
            time_clamped = time
        time_bins = discretize_time(time_clamped, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])

        loss = nll_logistic_hazard(log_hz_k, y, time_bins)
        
        loss = loss.to(x.device)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)

        return loss

    def validation_step(self, batch, batch_idx):
        x, y, time, demographic_targets = batch

        log_hz_k = self(x, y, time, demographic_targets)

        if self.time_range_years is not None:
            if y.ndim == 2:
                time = time.unsqueeze(1)
            y = torch.where(time > self.time_range_years[1]*365.25, torch.zeros_like(y), y)
            time_clamped = torch.clamp(time, min=self.time_range_years[0]*365.25, max=self.time_range_years[1]*365.25)
        else:
            time_clamped = time
        time_bins = discretize_time(time_clamped, bin_years=self.discrete_time_bins, min_year=self.time_range_years[0], max_year=self.time_range_years[1])
        loss = nll_logistic_hazard(log_hz_k, y, time_bins)

        loss = loss.to(x.device)
        log_hz_k = log_hz_k.to(x.device)
        self.log('val_loss', loss, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        
        self.events.append(y.bool().detach().cpu())
        self.times.append(time.detach().cpu())
        self.preds.append(log_hz_k.detach())
        return loss

    def on_validation_epoch_end(self):

        preds = torch.cat(self.preds).squeeze()
        preds = torch.sigmoid(preds)
        
        og_device = preds.device.type

        preds = preds.cpu().to(torch.float32).numpy()
        events = torch.cat(self.events).squeeze().numpy()
        times = torch.cat(self.times).squeeze().numpy()
        new_time = self.new_time.cpu().numpy()
        evaluate_risk_years = self.evaluate_risk_years.cpu().numpy()
        
        preds = 1 - hazards_to_survival(preds)

        try:
            min_time, max_time = times.min(), times.max()
            if new_time.min() < min_time or new_time.max() > max_time:
                warnings.warn(f"Some evaluation times are outside the range of observed times. Clipping new_time from [{new_time.min()}, {new_time.max()}] to [{min_time}, {max_time}]")
                idx_ = np.where((new_time >= min_time) & (new_time <= max_time))[0]
                preds = preds[:, idx_]
                new_time = new_time[idx_]
                evaluate_risk_years = evaluate_risk_years[idx_]

            min_first_event_time = times[events == True].min()

            idx_ = np.where((new_time >= min_first_event_time))[0]
            preds = preds[:, idx_]
            new_time = new_time[idx_]
            evaluate_risk_years = evaluate_risk_years[idx_]
            
            tau = new_time[-1]
            print(f"Events shape: {events.shape}, sum: {events.sum()}")
            print(f"Times shape: {times.shape}, range: [{times.min()}, {times.max()}]")
            print(f"Predictions shape: {preds.shape}, range: [{preds.min()}, {preds.max()}]")
        except Exception as e:
            print(f"IPCW or time clipping error: {str(e)}")


        try:
            train_events = torch.cat(self.train_ipcw_events).squeeze().numpy()
            train_times = torch.cat(self.train_ipcw_times).squeeze().numpy()

            
            train_structured = Surv.from_arrays(event=train_events, time=train_times)
            val_structured = Surv.from_arrays(event=events, time=times)
        except Exception as e:
            print(f"Structured array creation error: {str(e)}")
        try:
            auc_all, mean_auc = cumulative_dynamic_auc(train_structured, val_structured, preds, new_time)
            auc_all = torch.tensor(auc_all, device=og_device)
            mean_auc = torch.tensor(mean_auc, device=og_device)
        except Exception as e:
            print(f"AUC calculation error: {str(e)}")
            auc_all = torch.tensor([0.0 for _ in evaluate_risk_years], device=og_device)
            mean_auc = torch.tensor(0.0, device=og_device)
        finally:
            self.log_dict({f"{y}year_auc_ipcw": score for y, score in zip(evaluate_risk_years, auc_all)}, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            self.log("mean_ipcw_auc", mean_auc, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        try:
            cindex = concordance_index_ipcw(train_structured, val_structured, preds[:,-1], tau=tau)[0]
            cindex = torch.tensor(cindex, device=og_device)
            self.log("cindex_ipcw", cindex, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"C-index calculation error: {str(e)}")
        try:
            preds = 1 - preds
            brier = integrated_brier_score(train_structured, val_structured, preds, new_time)
            brier = torch.tensor(brier, device=og_device)
            self.log("brier_score", brier, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"Brier calculation error: {str(e)}")
        try:
            all_score = mean_auc + (1 - brier) + cindex
            self.log("all_score", all_score, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
        except Exception as e:
            print(f"All score calculation error: {str(e)}")
        self.preds.clear()
        self.events.clear()
        self.times.clear()

    def on_validation_epoch_start(self):
        torch.cuda.empty_cache()

    def configure_optimizers(self):
        param_groups = [
             {
                'params': (self.time_bias)
            },
            {
                'params': (p for n, p in self.demographic_effects.named_parameters()) 
            },
            {
                'params': (p for n, p in self.age_time_interaction.named_parameters())
            }
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, total_steps=self.trainer.estimated_stepping_batches, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()